<a href="https://colab.research.google.com/github/Chetalam/cnn-skin-cancer-classification/blob/main/notebooks/03_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/cnn-skin-cancer-classification"
)
DB_PATH = PROJECT_ROOT / "Database" / "skin_cancer.db"

if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)

with sqlite3.connect(
    DB_PATH.as_uri() + "?mode=ro", uri=True
) as conn:
    df = pd.read_sql_query("SELECT * FROM images", conn)

print("Records loaded:", len(df))

print("\nSex/gender values by dataset:")
display(pd.crosstab(
    df["sex"].fillna("(missing)"),
    df["dataset"]
))

print("\nMissing values by dataset:")
display(
    df.groupby("dataset")[
        ["patient_id", "lesion_id", "age", "sex", "localization"]
    ].agg(lambda column: column.isna().sum())
)

print("\nAge range by dataset:")
display(df.groupby("dataset")["age"].agg(["min", "max"]))

Mounted at /content/drive
Records loaded: 2716

Sex/gender values by dataset:


dataset,HAM10000,PAD-UFES-20
sex,,
FEMALE,0,524
MALE,0,565
female,621,0
male,1006,0



Missing values by dataset:


,patient_id,lesion_id,age,sex,localization
dataset,,,,,
HAM10000,1627,0,2,0,0
PAD-UFES-20,0,0,0,0,0



Age range by dataset:


,min,max
dataset,,
HAM10000,5.0,85.0
PAD-UFES-20,21.0,94.0


In [2]:
clean = df.copy()

# Trim text and convert blank strings to missing values.
text_columns = [
    "dataset", "image_id", "lesion_id", "patient_id",
    "original_label", "class_label", "sex",
    "localization", "image_path"
]

for column in text_columns:
    clean[column] = clean[column].astype("string").str.strip()
    clean[column] = clean[column].replace("", pd.NA)

clean["sex"] = clean["sex"].str.lower()
clean["age"] = pd.to_numeric(clean["age"], errors="coerce")

if not clean["sex"].isin(["female", "male"]).all():
    raise ValueError("Unexpected sex labels; inspect before continuing.")

if (clean["age"].dropna() < 0).any():
    raise ValueError("Negative ages found; inspect before continuing.")

required = ["dataset", "image_id", "class_label", "image_path"]
if clean[required].isna().any().any():
    raise ValueError("Missing required image metadata.")

if clean.duplicated(["dataset", "image_id"]).any():
    raise ValueError("Repeated dataset/image identifiers.")

print("Records after metadata standardisation:", len(clean))
display(pd.crosstab(clean["sex"], clean["dataset"]))
print("Missing ages retained:", clean["age"].isna().sum())

Records after metadata standardisation: 2716


dataset,HAM10000,PAD-UFES-20
sex,,
female,621,524
male,1006,565


Missing ages retained: 2


In [5]:
import hashlib
from PIL import Image, ImageOps

hashes = []

for number, row in enumerate(
    clean.itertuples(index=False), start=1
):
    with Image.open(row.image_path) as image:
        rgb = ImageOps.exif_transpose(image).convert("RGB")

        digest = hashlib.sha256()
        digest.update(f"{rgb.width}x{rgb.height}:RGB:".encode())
        digest.update(rgb.tobytes())
        hashes.append(digest.hexdigest())

    if number % 250 == 0 or number == len(clean):
        print(f"Checked {number}/{len(clean)} images")

clean["image_hash"] = hashes

REPORT_ROOT = PROJECT_ROOT / "Reports" / "Cleaning"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

duplicate_images = clean[
    clean.duplicated("image_hash", keep=False)
].sort_values("image_hash")

duplicate_images.to_csv(
    REPORT_ROOT / "exact_duplicate_images.csv", index=False
)

print("Rows involved in exact image duplicates:", len(duplicate_images))
display(duplicate_images[
    ["dataset", "image_id", "class_label", "image_hash"]
])

# Preserve all hashed records before filtering.
hashed_records = clean.copy()

# Quarantine every image whose identical pixels have conflicting labels.
label_counts = hashed_records.groupby(
    "image_hash"
)["class_label"].nunique()

conflicting_hashes = label_counts[label_counts > 1].index

conflicts = hashed_records[
    hashed_records["image_hash"].isin(conflicting_hashes)
].copy()

conflicts["exclusion_reason"] = "Identical pixels; conflicting class labels"
conflicts.to_csv(
    REPORT_ROOT / "conflicting_image_labels.csv",
    index=False
)

eligible = hashed_records[
    ~hashed_records["image_hash"].isin(conflicting_hashes)
].sort_values(["dataset", "image_id"]).copy()

# Retain one representative of each remaining identical image.
duplicate_mask = eligible.duplicated("image_hash", keep="first")
removed_duplicates = eligible.loc[duplicate_mask].copy()

removed_duplicates["exclusion_reason"] = (
    "Identical pixels and same class; representative retained"
)
removed_duplicates.to_csv(
    REPORT_ROOT / "removed_same_label_duplicates.csv",
    index=False
)

clean = eligible.loc[~duplicate_mask].reset_index(drop=True)

assert not clean["image_hash"].duplicated().any()
assert len(hashed_records) == (
    len(conflicts) + len(removed_duplicates) + len(clean)
)

print("Original records:", len(hashed_records))
print("Conflicting-label records excluded:", len(conflicts))
print("Same-label duplicate records excluded:", len(removed_duplicates))
print("Clean records retained:", len(clean))

display(clean["class_label"].value_counts())

Checked 250/2716 images
Checked 500/2716 images
Checked 750/2716 images
Checked 1000/2716 images
Checked 1250/2716 images
Checked 1500/2716 images
Checked 1750/2716 images
Checked 2000/2716 images
Checked 2250/2716 images
Checked 2500/2716 images
Checked 2716/2716 images
Rows involved in exact image duplicates: 14


,dataset,image_id,class_label,image_hash
1950,PAD-UFES-20,PAT_570_1085_429.png,BCC,31d718c275c95564e0b9177799cbe7011550d6ff19a968...
2325,PAD-UFES-20,PAT_570_1084_939.png,SCC,31d718c275c95564e0b9177799cbe7011550d6ff19a968...
1976,PAD-UFES-20,PAT_311_666_191.png,SCC,442001df656f2a5fde47e3e384012ff14280a848e8731c...
2387,PAD-UFES-20,PAT_311_667_416.png,BCC,442001df656f2a5fde47e3e384012ff14280a848e8731c...
1635,PAD-UFES-20,PAT_380_1540_959.png,SCC,8226b12cd6db365aa8c0b3e686d54eef63572a77e796d4...
2155,PAD-UFES-20,PAT_380_773_362.png,BCC,8226b12cd6db365aa8c0b3e686d54eef63572a77e796d4...
2573,PAD-UFES-20,PAT_570_1084_637.png,SCC,844ac7e5592a57056f2e27f905a4d127596a3b23140aed...
2574,PAD-UFES-20,PAT_570_1085_344.png,BCC,844ac7e5592a57056f2e27f905a4d127596a3b23140aed...
1642,PAD-UFES-20,PAT_302_651_529.png,BCC,b1f7f92fffcfef6cec0fdfac907e7fe339da698b6e9a56...
2442,PAD-UFES-20,PAT_302_650_477.png,SCC,b1f7f92fffcfef6cec0fdfac907e7fe339da698b6e9a56...


Original records: 2716
Conflicting-label records excluded: 12
Same-label duplicate records excluded: 1
Clean records retained: 2703


,count
class_label,
BCC,1353
Melanoma,1165
SCC,185


In [6]:
HAM_ROOT = PROJECT_ROOT / "Datasets" / "HAM10000"
ham_metadata = pd.read_csv(HAM_ROOT / "HAM10000_metadata.csv")

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

ham_files = [
    path for path in HAM_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in extensions
]

reference_ids = set(
    ham_metadata["image_id"].dropna().astype(str).str.strip()
)
actual_ids = {path.stem for path in ham_files}

unreferenced = [
    str(path) for path in ham_files
    if path.stem not in reference_ids
]
missing_ids = sorted(reference_ids - actual_ids)

print("HAM metadata rows:", len(ham_metadata))
print("HAM image files:", len(ham_files))
print("Files not referenced by metadata:", len(unreferenced))
print("Metadata image IDs not found:", len(missing_ids))

for path in unreferenced:
    print(path)

display(pd.DataFrame({"image_id": missing_ids}))

pd.DataFrame({"image_path": unreferenced}).to_csv(
    REPORT_ROOT / "ham_unreferenced_files.csv", index=False
)
pd.DataFrame({"image_id": missing_ids}).to_csv(
    REPORT_ROOT / "ham_missing_files.csv", index=False
)

HAM metadata rows: 10015
HAM image files: 10033
Files not referenced by metadata: 18
Metadata image IDs not found: 0
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0028746 (1).jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0028750 (1).jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0028859 (1).jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0026510 (1).jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0026513 (1).jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0026516 (1).jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0026223 (1).jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000

,image_id


## HAM10000 file audit

HAM10000 contains 10,033 image files and 10,015 metadata records.
All metadata-referenced image IDs were located.

The 18 unreferenced files have "(1)" filename suffixes, suggesting
additional copies. They were excluded from database ingestion.
Original files remain unchanged.

In [7]:
CLEAN_ROOT = PROJECT_ROOT / "Datasets" / "Cleaned"
CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

# Verify the manifest before exporting.
assert not clean.empty
assert not clean["image_hash"].duplicated().any()
assert not clean.duplicated(["dataset", "image_id"]).any()
assert clean["class_label"].isin(["Melanoma", "BCC", "SCC"]).all()
assert clean["image_path"].map(lambda p: Path(p).is_file()).all()

manifest_path = CLEAN_ROOT / "clean_metadata.csv"
clean.to_csv(manifest_path, index=False)

class_counts = (
    clean["class_label"].value_counts()
    .reindex(["Melanoma", "BCC", "SCC"], fill_value=0)
    .rename_axis("class_label")
    .reset_index(name="image_count")
)
class_counts.to_csv(
    REPORT_ROOT / "clean_class_distribution.csv", index=False
)

print("DATA CLEANING EXPORT COMPLETE")
print("Original selected records:", len(df))
print("Conflicting-label records excluded:", len(conflicts))
print("Same-label duplicate records excluded:", len(removed_duplicates))
print("Clean records:", len(clean))
print("Missing ages retained:", clean["age"].isna().sum())
print("Manifest saved to:", manifest_path)
display(class_counts)

DATA CLEANING EXPORT COMPLETE
Original selected records: 2716
Conflicting-label records excluded: 12
Same-label duplicate records excluded: 1
Clean records: 2703
Missing ages retained: 2
Manifest saved to: /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/Cleaned/clean_metadata.csv


,class_label,image_count
0,Melanoma,1165
1,BCC,1353
2,SCC,185


## Cleaning summary

- Original selected records: 2,716.
- Standardised sex/gender labels to lowercase.
- Excluded 12 records with identical pixels and conflicting labels,
  pending source verification.
- Removed one redundant same-label image record.
- Retained two missing ages without inventing values.
- Final manifest: 2,703 images — 1,165 Melanoma, 1,353 BCC and 185 SCC.
- Original database and image files remain unchanged.
- Clean manifest saved to Datasets/Cleaned/clean_metadata.csv.

Near-duplicate detection and grouped train/validation/test splitting
remain for the model preparation stage.